# Cleaning Order Reviews Dataset:
---

In [0]:
# Loading the dataset
df_reviews = spark.table("project.bronze.bronze_order_reviews")
display(df_reviews.head(10))

review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,ingestion_timestamp
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,null,null,2018-01-18 00:00:00,2018-01-18 21:46:59,2026-03-09T13:55:44.083Z
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,null,null,2018-03-10 00:00:00,2018-03-11 03:05:13,2026-03-09T13:55:44.083Z
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,null,null,2018-02-17 00:00:00,2018-02-18 14:36:24,2026-03-09T13:55:44.083Z
e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,null,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06,2026-03-09T13:55:44.083Z
f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,null,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01 00:00:00,2018-03-02 10:26:53,2026-03-09T13:55:44.083Z
15197aa66ff4d0650b5434f1b46cda19,b18dcdf73be66366873cd26c5724d1dc,1,null,null,2018-04-13 00:00:00,2018-04-16 00:39:37,2026-03-09T13:55:44.083Z
07f9bee5d1b850860defd761afa7ff16,e48aa0d2dcec3a2e87348811bcfdf22b,5,null,null,2017-07-16 00:00:00,2017-07-18 19:30:34,2026-03-09T13:55:44.083Z
7c6400515c67679fbee952a7525281ef,c31a859e34e3adac22f376954e19b39d,5,null,null,2018-08-14 00:00:00,2018-08-14 21:36:06,2026-03-09T13:55:44.083Z
a3f6f7f6f433de0aefbb97da197c554c,9c214ac970e84273583ab523dfafd09b,5,null,null,2017-05-17 00:00:00,2017-05-18 12:05:37,2026-03-09T13:55:44.083Z
8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recomendo,aparelho eficiente. no site a marca do aparelho esta impresso como 3desinfector e ao chegar esta com outro nome...atualizar com a marca correta uma vez que é o mesmo aparelho,2018-05-22 00:00:00,2018-05-23 16:45:47,2026-03-09T13:55:44.083Z


### Drop the unnecessary columns:

In [0]:
df_reviews_new = df_reviews.drop("review_id", "review_comment_title", "review_comment_message")
display(df_reviews_new.limit(5))

order_id,review_score,review_creation_date,review_answer_timestamp,ingestion_timestamp
73fc7af87114b39712e6da79b0a377eb,4,2018-01-18 00:00:00,2018-01-18 21:46:59,2026-03-09T13:55:44.083Z
a548910a1c6147796b98fdf73dbeba33,5,2018-03-10 00:00:00,2018-03-11 03:05:13,2026-03-09T13:55:44.083Z
f9e4b658b201a9f2ecdecbb34bed034b,5,2018-02-17 00:00:00,2018-02-18 14:36:24,2026-03-09T13:55:44.083Z
658677c97b385a9be170737859d3511b,5,2017-04-21 00:00:00,2017-04-21 22:02:06,2026-03-09T13:55:44.083Z
8e6bfb81e283fa7e4f11123a3fb894f1,5,2018-03-01 00:00:00,2018-03-02 10:26:53,2026-03-09T13:55:44.083Z


In [0]:
# print schema for any mismatches
df_reviews_new.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- review_score: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



In [0]:
# calculate the unique values 
for i in ["review_score","review_creation_date","review_answer_timestamp"]:
    display(df_reviews_new.groupBy(i).count().orderBy("count", ascending=False))

review_score,count
5,57328
4,19142
1,11424
3,8179
2,3151
null,2380
2018-05-19 00:00:00,5
2017-12-29 00:00:00,4
2018-02-01 00:00:00,4
2018-01-04 00:00:00,4


review_creation_date,count
null,8764
2017-12-19 00:00:00,445
2018-08-28 00:00:00,434
2017-12-20 00:00:00,425
2018-05-19 00:00:00,417
2018-08-14 00:00:00,405
2018-05-22 00:00:00,405
2018-03-29 00:00:00,404
2018-05-15 00:00:00,399
2018-05-04 00:00:00,393


review_answer_timestamp,count
null,8785
2017-06-15 23:21:05,4
2018-03-30 00:29:09,3
2018-08-31 22:29:09,3
2017-08-30 11:18:29,3
2018-06-17 23:48:04,3
2017-02-21 23:30:22,3
2017-09-05 12:12:51,3
2017-04-13 21:12:43,3
2017-09-08 00:24:52,3


### Observation: 
- `review_score`, `review_creation_timestamp` and `review_answer_timestamp` are in string format. 
- timestamp is inside the `review_score`
- text is inside `review_creation_timestamp`


### Below is a robust preprocessing function that safely handles:

- corrupted review_score
- text inside review_creation_date
- text inside review_answer_timestamp
- malformed rows

In [0]:
from pyspark.sql.functions import col, when, to_timestamp, expr

def preprocess_order_reviews(df_reviews):

    df_reviews_silver = (
        df_reviews
        
        # Keep only valid review scores (1–5)
        .withColumn(
            "review_score",
            when(col("review_score").isin("1","2","3","4","5"),
                 col("review_score").cast("int"))
        )
        .filter(col("review_score").isNotNull())

        # Safe timestamp conversion (invalid values → NULL)
        .withColumn(
            "review_creation_date",
            expr("try_cast(review_creation_date as timestamp)")
        )

        .withColumn(
            "review_answer_timestamp",
            expr("try_cast(review_answer_timestamp as timestamp)")
        )

        # # Remove duplicate reviews
        # .dropDuplicates(["order_id"])
    )

    return df_reviews_silver

In [0]:
df_reviews_silver = preprocess_order_reviews(df_reviews_new)

display(df_reviews_silver.head(10))

order_id,review_score,review_creation_date,review_answer_timestamp,ingestion_timestamp
73fc7af87114b39712e6da79b0a377eb,4,2018-01-18T00:00:00.000Z,2018-01-18T21:46:59.000Z,2026-03-09T13:55:44.083Z
a548910a1c6147796b98fdf73dbeba33,5,2018-03-10T00:00:00.000Z,2018-03-11T03:05:13.000Z,2026-03-09T13:55:44.083Z
f9e4b658b201a9f2ecdecbb34bed034b,5,2018-02-17T00:00:00.000Z,2018-02-18T14:36:24.000Z,2026-03-09T13:55:44.083Z
658677c97b385a9be170737859d3511b,5,2017-04-21T00:00:00.000Z,2017-04-21T22:02:06.000Z,2026-03-09T13:55:44.083Z
8e6bfb81e283fa7e4f11123a3fb894f1,5,2018-03-01T00:00:00.000Z,2018-03-02T10:26:53.000Z,2026-03-09T13:55:44.083Z
b18dcdf73be66366873cd26c5724d1dc,1,2018-04-13T00:00:00.000Z,2018-04-16T00:39:37.000Z,2026-03-09T13:55:44.083Z
e48aa0d2dcec3a2e87348811bcfdf22b,5,2017-07-16T00:00:00.000Z,2017-07-18T19:30:34.000Z,2026-03-09T13:55:44.083Z
c31a859e34e3adac22f376954e19b39d,5,2018-08-14T00:00:00.000Z,2018-08-14T21:36:06.000Z,2026-03-09T13:55:44.083Z
9c214ac970e84273583ab523dfafd09b,5,2017-05-17T00:00:00.000Z,2017-05-18T12:05:37.000Z,2026-03-09T13:55:44.083Z
b9bf720beb4ab3728760088589c62129,4,2018-05-22T00:00:00.000Z,2018-05-23T16:45:47.000Z,2026-03-09T13:55:44.083Z


In [0]:
# calculate the unique values 
for i in ["review_score","review_creation_date","review_answer_timestamp"]:
    display(df_reviews_silver.groupBy(i).count().orderBy("count", ascending=False))

review_score,count
5,57019
4,19044
1,11353
3,8124
2,3133


review_creation_date,count
null,3888
2017-12-19T00:00:00.000Z,442
2018-08-28T00:00:00.000Z,434
2017-12-20T00:00:00.000Z,420
2018-05-19T00:00:00.000Z,415
2018-05-22T00:00:00.000Z,405
2018-08-14T00:00:00.000Z,405
2018-03-29T00:00:00.000Z,402
2018-05-15T00:00:00.000Z,398
2018-05-04T00:00:00.000Z,393


review_answer_timestamp,count
null,3841
2017-06-15T23:21:05.000Z,4
2017-08-30T11:18:29.000Z,3
2017-11-02T00:11:06.000Z,3
2017-04-13T21:12:43.000Z,3
2017-12-09T15:28:29.000Z,3
2017-09-08T00:24:52.000Z,3
2018-06-17T23:48:04.000Z,3
2018-09-03T09:33:00.000Z,3
2018-08-19T22:35:54.000Z,3


In [0]:
# saving the cleaned data
df_reviews_silver.write.format('delta').mode('overwrite').saveAsTable("project.silver.silver_reviews")

# Cleaning Orders Dataset:
---

In [0]:
df_orders = spark.table("project.bronze.bronze_orders")
display(df_orders.limit(10))

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,ingestion_timestamp
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z,2026-03-09T13:55:29.448Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z,2026-03-09T13:55:29.448Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z,2026-03-09T13:55:29.448Z
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z,2026-03-09T13:55:29.448Z
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z,2026-03-09T13:55:29.448Z
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09T21:57:05.000Z,2017-07-09T22:10:13.000Z,2017-07-11T14:58:04.000Z,2017-07-26T10:57:55.000Z,2017-08-01T00:00:00.000Z,2026-03-09T13:55:29.448Z
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11T12:22:08.000Z,2017-04-13T13:25:17.000Z,null,null,2017-05-09T00:00:00.000Z,2026-03-09T13:55:29.448Z
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16T13:10:30.000Z,2017-05-16T13:22:11.000Z,2017-05-22T10:07:46.000Z,2017-05-26T12:55:51.000Z,2017-06-07T00:00:00.000Z,2026-03-09T13:55:29.448Z
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23T18:29:09.000Z,2017-01-25T02:50:47.000Z,2017-01-26T14:16:31.000Z,2017-02-02T14:08:10.000Z,2017-03-06T00:00:00.000Z,2026-03-09T13:55:29.448Z
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29T11:55:02.000Z,2017-07-29T12:05:32.000Z,2017-08-10T19:45:24.000Z,2017-08-16T17:14:30.000Z,2017-08-23T00:00:00.000Z,2026-03-09T13:55:29.448Z


In [0]:
# Checking the schema for invalid datatype
df_orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



In [0]:
# calculating null values
{col: df_orders.filter(df_orders[col].isNull()).count() for col in df_orders.columns}

{'order_id': 0,
 'customer_id': 0,
 'order_status': 0,
 'order_purchase_timestamp': 0,
 'order_approved_at': 160,
 'order_delivered_carrier_date': 1783,
 'order_delivered_customer_date': 2965,
 'order_estimated_delivery_date': 0,
 'ingestion_timestamp': 0}

### Observation: 
- `order_approved_at` has 160 null values
- `order_delivered_carrier_date` has 1783 null values
- ` order_delivered_customer_date` has 2965 null values

Filling the missing `approved_at` timestamps with `purchase_timestamps`

In [0]:
from pyspark.sql.functions import col, coalesce

# Fill missing approval timestamps with purchase timestamp
df_orders_new = df_orders.withColumn(
    "order_approved_at",
    coalesce(col("order_approved_at"), col("order_purchase_timestamp"))
)


# # Keep only completed orders (removes most delivery nulls)
# df_orders = df_orders_new.filter(col("order_status") == "delivered")


# Remove duplicate orders (safe for pipelines)
df_orders_new = df_orders_new.dropDuplicates(["order_id"])

In [0]:
# Check for null values again
{col: df_orders_new.filter(df_orders_new[col].isNull()).count() for col in df_orders_new.columns}

{'order_id': 0,
 'customer_id': 0,
 'order_status': 0,
 'order_purchase_timestamp': 0,
 'order_approved_at': 0,
 'order_delivered_carrier_date': 1783,
 'order_delivered_customer_date': 2965,
 'order_estimated_delivery_date': 0,
 'ingestion_timestamp': 0}

***Checking the unique values in `order_status`***

In [0]:
df_orders_new.groupBy("order_status").count().show()

+------------+-----+
|order_status|count|
+------------+-----+
|   delivered|96478|
|    invoiced|  314|
|     shipped| 1107|
|  processing|  301|
| unavailable|  609|
|    canceled|  625|
|     created|    5|
|    approved|    2|
+------------+-----+



### For each distinct value in `order_status`, calculate the null_delivered_customer_date_count`

In [0]:
order_status_values = [row["order_status"] for row in df_orders_new.select("order_status").distinct().collect()]

for status in order_status_values:
    count = df_orders_new.filter(
        (col("order_status") == status) &
        (col("order_delivered_customer_date").isNull())
    ).count()
    print({"order_status": status, "null_delivered_customer_date_count": count})

{'order_status': 'delivered', 'null_delivered_customer_date_count': 8}
{'order_status': 'invoiced', 'null_delivered_customer_date_count': 314}
{'order_status': 'shipped', 'null_delivered_customer_date_count': 1107}
{'order_status': 'processing', 'null_delivered_customer_date_count': 301}
{'order_status': 'unavailable', 'null_delivered_customer_date_count': 609}
{'order_status': 'canceled', 'null_delivered_customer_date_count': 619}
{'order_status': 'created', 'null_delivered_customer_date_count': 5}
{'order_status': 'approved', 'null_delivered_customer_date_count': 2}


In [0]:
order_status_values = [row["order_status"] for row in df_orders_new.select("order_status").distinct().collect()]

for status in order_status_values:
    count = df_orders_new.filter(
        (col("order_status") == status) &
        (col("order_delivered_carrier_date").isNull())
    ).count()
    print({"order_status": status, "null_delivered_customer_date_count": count})

{'order_status': 'delivered', 'null_delivered_customer_date_count': 2}
{'order_status': 'invoiced', 'null_delivered_customer_date_count': 314}
{'order_status': 'shipped', 'null_delivered_customer_date_count': 0}
{'order_status': 'processing', 'null_delivered_customer_date_count': 301}
{'order_status': 'unavailable', 'null_delivered_customer_date_count': 609}
{'order_status': 'canceled', 'null_delivered_customer_date_count': 550}
{'order_status': 'created', 'null_delivered_customer_date_count': 5}
{'order_status': 'approved', 'null_delivered_customer_date_count': 2}


In [0]:
# saving the cleaned data
df_orders_new.write.format('delta').mode('overwrite').saveAsTable("project.silver.silver_orders")